# ICOS restructured NetCDF — variable explorer

Demonstrates reading the multi-group CF-1.12 NetCDF4 file produced by
`fluxnet_restructure.py` and plotting key variables from each group.

| Group | Time resolution | Key content |
|---|---|---|
| `/` (root) | Half-hourly | Met forcing, NEE, GPP, RECO, LE, H, soil |
| `/fluxnet_dd` | Daily | Same variables aggregated to daily |
| `/fluxnet_mm` | Monthly | Monthly means |
| `/fluxnet_yy` | Annual | Annual sums / means |

In [ ]:
import pathlib
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone

FILE = pathlib.Path("ICOSETC_SE-Svb_INTERIM_restructured.nc")

def decode_time(grp):
    """Return an array of Python datetimes from a CF time variable."""
    tv = grp["time"]
    return nc.num2date(tv[:], tv.units, tv.calendar if hasattr(tv, "calendar") else "standard",
                       only_use_cftime_datetimes=False)

def mask(arr, fv=None):
    """Return float64 array with fill values → NaN.

    Handles netCDF4 MaskedArrays directly so that masked entries become NaN
    instead of leaking through as the raw _FillValue (≈9.97e+36).
    """
    raw = arr[:]
    if isinstance(raw, np.ma.MaskedArray):
        a = np.ma.filled(raw, np.nan).astype(float)
    else:
        a = np.array(raw, dtype=float)
    if fv is None and hasattr(arr, "_FillValue"):
        fv = float(arr._FillValue)
    if fv is not None:
        a[a == fv] = np.nan
    return a

ds = nc.Dataset(FILE)
site_id = getattr(ds, "site_id", getattr(ds, "station_name", ""))
print("Groups :", list(ds.groups.keys()))
print("Station :", getattr(ds, "station_name", ds.filepath()))
print("Site    :", site_id)
print("Lat/Lon :", getattr(ds, "geospatial_lat", "—"), getattr(ds, "geospatial_lon", "—"))
print("Height  :", getattr(ds, "station_elevation", "—"), getattr(ds, "station_elevation_units", "—"))
print("Ecotype :", getattr(ds, "ecosystem", "—"))

## 1 · Half-hourly meteorology (root group)

Air temperature and incoming shortwave radiation — the two primary drivers of ecosystem exchange.

In [ ]:
t_hh  = decode_time(ds)
ta    = mask(ds["TA_F"])          # Air temperature, gap-filled  [°C]
sw_in = mask(ds["SW_IN_F"])       # Shortwave radiation in       [W m⁻²]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle(f"{site_id} — Half-hourly meteorology (root group)", fontweight="bold")

axes[0].plot(t_hh, ta, lw=0.4, color="tab:red")
axes[0].set_ylabel("Air temperature (°C)")
axes[0].axhline(0, color="k", lw=0.5, ls="--")

axes[1].plot(t_hh, sw_in, lw=0.4, color="goldenrod")
axes[1].set_ylabel("SW$_{in}$ (W m⁻²)")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 2 · Daily carbon fluxes (`/fluxnet_dd`)

**Top panel** — NEE with joint uncertainty envelope (VUT threshold, REF variant).  
**Bottom panel** — GPP and RECO from both partitioning methods (NT = night-time, DT = day-time).

In [ ]:
dd = ds["fluxnet_dd"]
t_dd = decode_time(dd)

# Dimension indices
i_vut, i_ref = 1, 0          # ustar_threshold: CUT=0, VUT=1
i_nt,  i_dt  = 0, 1          # partition_method: NT=0, DT=1

nee      = mask(dd["NEE"]         [:, i_vut, i_ref])
nee_unc  = mask(dd["NEE_JOINTUNC"][:, i_vut, i_ref])
gpp_nt   = mask(dd["GPP"]         [:, i_vut, i_nt,  i_ref])
gpp_dt   = mask(dd["GPP"]         [:, i_vut, i_dt,  i_ref])
reco_nt  = mask(dd["RECO"]        [:, i_vut, i_nt,  i_ref])
reco_dt  = mask(dd["RECO"]        [:, i_vut, i_dt,  i_ref])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f"{site_id} — Daily carbon fluxes (/fluxnet_dd, VUT threshold, REF variant)", fontweight="bold")

# NEE + uncertainty band
ax = axes[0]
ax.fill_between(t_dd, nee - nee_unc, nee + nee_unc, alpha=0.25, color="tab:blue", label="Joint unc.")
ax.plot(t_dd, nee, lw=1, color="tab:blue", label="NEE VUT REF")
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.set_ylabel("NEE (µmol m⁻² s⁻¹)")
ax.legend(fontsize=8)

# GPP & RECO, both partition methods
ax = axes[1]
ax.plot(t_dd, gpp_nt,  lw=1, color="tab:green", label="GPP NT")
ax.plot(t_dd, gpp_dt,  lw=1, color="limegreen",  label="GPP DT", ls="--")
ax.plot(t_dd, reco_nt, lw=1, color="tab:brown",  label="RECO NT")
ax.plot(t_dd, reco_dt, lw=1, color="peru",        label="RECO DT", ls="--")
ax.set_ylabel("GPP / RECO (µmol m⁻² s⁻¹)")
ax.legend(fontsize=8, ncol=2)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 3 · Monthly energy balance & soil profiles (`/fluxnet_mm`)

**Top** — Latent heat (LE) and sensible heat (H) at the median energy-balance correction (p50).  
**Bottom** — Soil temperature and soil water content profiles across all measured layers.

In [ ]:
mm = ds["fluxnet_mm"]
t_mm = decode_time(mm)

# --- energy balance: pick p50 correction (dim ebc_correction, index 1 = p50) ---
le_var = mm["LE_CORR"] if "LE_CORR" in mm.variables else mm["LE_F_MDS"]
h_var  = mm["H_CORR"]  if "H_CORR"  in mm.variables else mm["H_F_MDS"]

def pick_p50(var):
    a = mask(var)
    return a[:, 1] if a.ndim == 2 else a   # index 1 = p50 if ebc_correction dim present

le = pick_p50(le_var)
h  = pick_p50(h_var)

# --- soil profiles: TS(time, soil_layer) and SWC(time, soil_layer) ---
ts_all  = mask(mm["TS"])  if "TS"  in mm.variables else None   # shape (time, soil_layer)
swc_all = mask(mm["SWC"]) if "SWC" in mm.variables else None

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f"{site_id} — Monthly energy balance & soil profiles (/fluxnet_mm)", fontweight="bold")

ax = axes[0]
ax.plot(t_mm, le, lw=1.5, color="tab:cyan",   label="LE (p50 EBC)")
ax.plot(t_mm, h,  lw=1.5, color="tab:orange", label="H  (p50 EBC)")
ax.axhline(0, color="k", lw=0.5, ls="--")
ax.set_ylabel("Energy flux (W m⁻²)")
ax.legend(fontsize=9)

ax = axes[1]
if ts_all is not None:
    n_ts = ts_all.shape[1]
    cmap_ts = plt.cm.get_cmap("YlOrRd", max(n_ts, 2))
    for i in range(n_ts):
        ax.plot(t_mm, ts_all[:, i], lw=1, color=cmap_ts(i / max(n_ts - 1, 1)),
                label=f"TS layer {i+1}")

if swc_all is not None:
    n_swc = swc_all.shape[1]
    cmap_swc = plt.cm.get_cmap("Blues", max(n_swc + 2, 3))
    for i in range(n_swc):
        ax.plot(t_mm, swc_all[:, i], lw=1, color=cmap_swc(i / max(n_swc - 1, 1) * 0.6 + 0.4),
                ls="--", label=f"SWC layer {i+1}")

ax.set_ylabel("Soil T (°C) / SWC (%)")
ax.legend(fontsize=7, ncol=3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 4 · Annual carbon budget (`/fluxnet_yy`)

Bar chart of annual NEE, GPP, and RECO (VUT threshold, REF variant, NT partitioning).  
Negative NEE = net carbon uptake (sink); positive = source.

In [ ]:
yy = ds["fluxnet_yy"]
t_yy = decode_time(yy)
years = np.array([t.year for t in t_yy])

i_vut, i_ref, i_nt = 1, 0, 0   # same as daily section

nee_y  = mask(yy["NEE"] [:, i_vut, i_ref])
gpp_y  = mask(yy["GPP"] [:, i_vut, i_nt, i_ref])
reco_y = mask(yy["RECO"][:, i_vut, i_nt, i_ref])

x = np.arange(len(years))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle(f"{site_id} — Annual carbon budget (/fluxnet_yy, VUT REF, NT partitioning)", fontweight="bold")

ax.bar(x - w, nee_y,  w, label="NEE",  color="tab:blue",  alpha=0.85)
ax.bar(x,     gpp_y,  w, label="GPP",  color="tab:green", alpha=0.85)
ax.bar(x + w, reco_y, w, label="RECO", color="tab:brown", alpha=0.85)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("gC m⁻² yr⁻¹")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()